# Notebook 29 — Deep Random Forest Research and Entry Robustness

**Purpose:** A larger, nested chronological Random Forest-only search motivated by Notebook 28's poor out-of-sample entry coverage. Research-only: no historical final holdout evaluation and no prospective ledger writes.

**Notebook 28 reference:** Accuracy policy ~42.31% versus ~41.95% majority baseline; entry policy averaged ~48.27% Up precision but had just **44** trades in fold 2 and **321** in fold 3, with a negative fold-1 return. This notebook explicitly diagnoses that instability.

**Protocol:** 3 outer expanding folds, 2 inner expanding folds; select settings and thresholds only on inner data; report outer outcomes once. All features are causal; target labels use the existing Q7 mapping. All results are *development* evidence, not independent post-selection validation.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
if not (ROOT/'src').exists():raise FileNotFoundError('Run from repository root or notebooks folder')
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.rf_deep_research import (run_rf_deep_research,candidate_grid,
    cost_sensitivity,daily_aggregated_returns,THRESHOLDS)
print('Project:',ROOT)

## 1. Experiment controls

The default grid is **36 configurations × 2 feature families × 2 sampling schemes × 2 inner folds × 3 outer folds = 864 inner fits**, plus up to six outer refits. Each fit uses up to 60,000 rows and 200 trees. This can be time-consuming and RAM-intensive; adjust `TRAIN_CAP`, `N_ESTIMATORS`, `SAMPLING_MODES`, or `CONFIGS` for a smoke run. Thresholds are evaluated from stored probabilities without refitting.

**Sampling experiment:** uniform history vs most recent history. This tests whether stale data are harming performance.

**Entry selection:** minimum coverage and trade count must hold in **each** inner fold. Candidate score rewards mean net returns and penalizes the worst fold. Outer coverage is never forced or optimized after the fact.

In [ ]:
N_OUTER=3
N_INNER=2
TRAIN_CAP=60000
N_ESTIMATORS=200
COST_BPS=10
MIN_ENTRIES=150
MIN_COVERAGE=.025
SAMPLING_MODES=('uniform','recent')
CONFIGS=candidate_grid()
OUTPUT=ROOT/'artifacts'/'rf_deep_research'/'1d'
print('RF configurations:',len(CONFIGS),'Inner fits:',len(CONFIGS)*2*len(SAMPLING_MODES)*N_INNER*N_OUTER)
print('Thresholds:',THRESHOLDS)

## 2. Load the existing labeled master

`load_master()` returns **four values**. The first is the DataFrame. Notebook 29 does not rebuild or overwrite any master artifact.

In [ ]:
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
paths=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
source=next((p for p in paths if all((p/f).exists() for f in required)),None)
if source is None:raise FileNotFoundError('Missing daily master artifacts; run Notebook 03')
master,latest,bundle,manifest=load_master(source)
assert isinstance(master,pd.DataFrame),type(master)
assert {'symbol','interval_code','bar_timestamp','next_bar_timestamp','next_return'}.issubset(master.columns)
print('Rows:',len(master),'Symbols:',master.symbol.nunique(),'Intervals:',master.interval_code.value_counts().to_dict())

## 3. Run nested Random Forest search

Expect a long runtime. The module writes intermediate result **tables only after completion**, never overwriting holdout or prediction ledgers.

In [ ]:
results,cutoff=run_rf_deep_research(master,n_outer=N_OUTER,n_inner=N_INNER,
    cap=TRAIN_CAP,n_estimators=N_ESTIMATORS,cost_bps=COST_BPS,
    min_count=MIN_ENTRIES,min_coverage=MIN_COVERAGE,
    sampling_modes=SAMPLING_MODES,configs=CONFIGS,output_dir=OUTPUT)
print('Development cutoff:',cutoff)
print('Saved results:',OUTPUT)

## 4. Outer-fold scorecard: accuracy and entry policy

**Do not** rank and deploy based on outer results; use these only to assess whether the inner-selected decision rule generalizes.

In [ ]:
outer=results['outer'];display(outer.round(5))
if not outer.empty:
    display(outer.groupby('objective').agg(folds=('fold','nunique'),accuracy=('accuracy','mean'),
        auc=('auc','mean'),baseline=('majority_baseline','mean'),precision=('precision','mean'),
        coverage=('coverage','mean'),mean_net=('mean_net','mean'),
        min_entries=('entry_count','min'),total_entries=('entry_count','sum')).round(5))
display(results['selections'].round(5))

## 5. Full inner search leaderboard and ablations

This is an exploratory summary of the nested *training* searches, not a second independent validation set. Compare unweighted vs balanced, leaf size, depth, max-features, sampling, and price vs price-volume.

In [ ]:
inner=results['inner']
if not inner.empty:
    base=inner[inner.threshold==.5]
    display(base.groupby(['features','sampling']).agg(accuracy=('accuracy','mean'),auc=('auc','mean'),
        logloss=('logloss','mean'),brier_up=('brier_up','mean')).round(5))
    display(base.groupby('config').agg(accuracy=('accuracy','mean'),auc=('auc','mean'),
        fit_seconds=('fit_seconds','mean')).sort_values('accuracy',ascending=False).head(25).round(5))
    display(base.groupby('outer_fold').fit_seconds.sum().rename('inner_fit_seconds'))

## 6. Signal concentration, tail risk, and transaction-cost sensitivity

The entry strategy must survive a realistic range of round-trip costs. Trades sharing a date are correlated, so we also report equal-weight *daily* results rather than treating all ticker trades as independent observations.

In [ ]:
trades=results['trades']
if not trades.empty:
    display(trades.groupby(['objective','fold']).agg(n=('net_return','size'),
        mean_net=('net_return','mean'),median_net=('net_return','median'),
        p01=('net_return',lambda x:x.quantile(.01)),p05=('net_return',lambda x:x.quantile(.05)),
        p95=('net_return',lambda x:x.quantile(.95)),
        win_rate=('net_return',lambda x:(x>0).mean())).round(5))
    display(cost_sensitivity(trades).round(5))
    daily=daily_aggregated_returns(trades)
    display(daily.groupby(['objective','fold']).agg(days=('date','nunique'),
        mean_daily_net=('mean_net','mean'),worst_daily_net=('mean_net','min'),
        mean_signals_per_day=('trades','mean')).round(5))
else:print('No entry signals passed the selected thresholds')

## 7. Calibration and feature stability

These are diagnostics on outer validation predictions, **not** a license to choose a new probability threshold using outer data.

In [ ]:
if not results['calibration'].empty:
    display(results['calibration'].groupby(['objective','bin_low','bin_high']).agg(
        n=('n','sum'),mean_p=('mean_probability','mean'),observed_up=('actual_up','mean'),
        mean_return=('mean_return','mean')).round(5))
if not results['importances'].empty:
    display(results['importances'].groupby(['objective','feature']).importance.agg(['mean','std','count'])
        .sort_values(['objective','mean'],ascending=[True,False]).groupby(level=0).head(20).round(5))
print('Outer refit timings:');display(results['timings'].round(2))

## 8. Research gate and Notebook 30 decision

An entry rule is **not** ready for shadow deployment unless each outer fold meets the minimum trade count and coverage, net returns are positive after costs in all three folds, and there is no serious calibration or concentration concern. Passing this diagnostic gate is necessary, not sufficient; the previously evaluated final holdout remains closed.

If no rule passes, consider *point-in-time sector membership, market regime context, and event/news features* before another expensive hyperparameter grid.

In [ ]:
if outer.empty:print('NO PROMOTION — no outer evaluation')
else:
    for objective,group in outer.groupby('objective'):
        passes=(len(group)==3 and (group.entry_count>=MIN_ENTRIES).all()
                and (group.coverage>=MIN_COVERAGE).all() and (group.mean_net>0).all())
        print(objective,':','PASSES PRELIMINARY DIAGNOSTIC — independent validation still required' if passes
              else 'NO PROMOTION — research only')
print('Historical final holdout and prospective prediction ledger unchanged.')